In [ ]:
import requests
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import nltk
import time


In [ ]:
nltk.download('punkt')  # Tải tokenizer nếu chưa có

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
BASE_URL = "https://nld.com.vn"
MAIN_URL = BASE_URL

headers = {
    'User-Agent': 'Mozilla/5.0'
}


In [ ]:
# B1. Tải trang chính
res = requests.get(MAIN_URL, headers=headers)
soup = BeautifulSoup(res.text, 'html.parser')


In [ ]:
# B2: tìm tất cả chuyên mục trong thẻ nav
nav= soup.find("div",class_="header__n-flex")

In [ ]:
all_a= nav.find_all("a",class_="nav-link")
print(all_a)

[<a class="nav-link" href="/" title="Trang chủ">
<svg fill="none" height="20" viewbox="0 0 20 20" width="20" xmlns="http://www.w3.org/2000/svg">
<path d="M17.7313 9.48406L10.5526 2.9429C10.2375 2.65574 9.76235 2.65577 9.44738 2.94287L2.26869 9.48409C2.01628 9.71409 1.93291 10.0685 2.05622 10.3869C2.17956 10.7053 2.47987 10.911 2.82134 10.911H3.9679V17.4648C3.9679 17.7246 4.17859 17.9353 4.43843 17.9353H8.37323C8.63307 17.9353 8.84376 17.7246 8.84376 17.4648V13.4855H11.1563V17.4648C11.1563 17.7246 11.367 17.9353 11.6268 17.9353H15.5615C15.8213 17.9353 16.032 17.7247 16.032 17.4648V10.911H17.1788C17.5202 10.911 17.8205 10.7052 17.9439 10.3869C18.067 10.0684 17.9837 9.71409 17.7313 9.48406Z" fill="white"></path>
<path d="M17.3875 2.33057H13.4375L17.9756 6.29351V2.89539C17.9756 2.58346 17.7123 2.33057 17.3875 2.33057Z" fill="white"></path>
</svg>
</a>, <a class="nav-link" href="/thoi-su.htm" title="Thời sự">Thời sự</a>, <a class="nav-link" href="/quoc-te.htm" title="Quốc tế">Quốc tế</a>, <

In [ ]:
x = ['/']
topic_links = [a['href'] for a in all_a if 'href' in a.attrs and a['href'] not in x]


In [ ]:
print(topic_links)

['/thoi-su.htm', '/quoc-te.htm', '/lao-dong.htm', '/ban-doc.htm', '/net-zero.htm', '/kinh-te.htm', '/suc-khoe.htm', '/giao-duc-khoa-hoc.htm', '/phap-luat.htm', '/van-hoa-van-nghe.htm', '/giai-tri.htm', '/the-thao.htm', '/ai-365.htm', 'https://phunu.nld.com.vn/', '/gia-dinh.htm', 'https://diaoc.nld.com.vn']


In [ ]:
topic_links

['/thoi-su.htm',
 '/quoc-te.htm',
 '/lao-dong.htm',
 '/ban-doc.htm',
 '/net-zero.htm',
 '/kinh-te.htm',
 '/suc-khoe.htm',
 '/giao-duc-khoa-hoc.htm',
 '/phap-luat.htm',
 '/van-hoa-van-nghe.htm',
 '/giai-tri.htm',
 '/the-thao.htm',
 '/ai-365.htm',
 'https://phunu.nld.com.vn/',
 '/gia-dinh.htm',
 'https://diaoc.nld.com.vn']

In [ ]:
pip install underthesea

In [ ]:
from underthesea import sent_tokenize


In [ ]:
# def extract_intro(text, max_sent=3):
#     try:
#         sents = sent_tokenize(text)
#         return " ".join(sents[:max_sent])
#     except Exception:
#         return ""

def extract_intro(text, max_sent=3):
    try:
        sents = sent_tokenize(text)
        return " ".join(sents[:max_sent])
    except Exception:
        return ""

In [ ]:
import os

In [ ]:
SAVE_DIR = "du_lieu_bao_nguoi_lao_dong"
os.makedirs(SAVE_DIR, exist_ok=True)

output_path = os.path.join(SAVE_DIR, "true.csv")

# Nếu chưa có file, tạo dòng header
if not os.path.exists(output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("title,link,summary,intro,label\n")

In [ ]:
# B3. Duyệt từng chuyên mục
i=0
for topic_link in topic_links:
    i+=1
    topic_url = topic_link if topic_link.startswith("http") else BASE_URL + topic_link
    print(f" Chuyên mục {i}: {topic_url}")
# oke
    try:
        topic_res = requests.get(topic_url, headers=headers)
        topic_soup = BeautifulSoup(topic_res.text, 'html.parser')

        # Tìm các bài viết — giả định thẻ <a class="story__title">
        articles = topic_soup.find_all("div", class_="box-category-item")
        print(len(articles))
        count = 0

        for article in articles:
            if count >=30:
                break
            count+=1
            h3 = article.find('a', {'data-linktype': 'newsdetail'})

            # title = soup.find('a', {'data-linktype': 'newsdetail'}).get_text(strip=True)
            # print(h3)
            a_tag = article.find("a", class_="box-category-link-with-avatar") if article else None
            if not a_tag or not a_tag.has_attr('href'):
                continue

            link = a_tag['href']
            if not link.startswith("http"):
                link = BASE_URL + link

            title = h3.get_text(strip=True)

            print(f" Link bài: {link}")
            print(f" Tiêu đề: {title}")

            #  Mở link và lấy nội dung, giống như các bước trước
            try:
                news_res = requests.get(link, headers=headers)
                news_soup = BeautifulSoup(news_res.text, 'html.parser')

                sumary_h2 = news_soup.find("h2", class_="detail-sapo")
                if not sumary_h2:
                    continue

                paragraphs = sumary_h2
                summary_text = " ".join(p.get_text(strip=True) for p in paragraphs)

                print(f" Tóm tắt: {summary_text}\n")

                content_div = news_soup.find("div", class_="detail-content")

                # print(content_div)
                if not content_div:
                    continue

                paragraphs = content_div.find_all("p")

                # print(paragraphs)

                # Nối tất cả nội dung thẻ <p> thành một đoạn văn
                text_content = " ".join([p.get_text(strip=True) for p in paragraphs])

                # Lấy phần giới thiệu
                intro = extract_intro(text_content, max_sent=3)
                # print(f" 3 câu đầu: {intro}\n")


                full_text = f"{title}. {summary_text} {intro}"
                print("Toàn bộ dữ liệu:",full_text)
                with open(output_path, "a", encoding="utf-8") as f:
                    f.write(f'"{full_text}"\n')
                # count+=1
                time.sleep(1)

            except Exception as e:
                print(f" Lỗi khi đọc bài: {e}")


    except Exception as e:
        print(f"Lỗi khi duyệt chuyên mục: {e}")


 Chuyên mục 1: https://nld.com.vn/thoi-su.htm
24
 Link bài: https://nld.com.vn/can-bo-cong-an-tu-vong-vi-tai-nan-giao-thong-hy-huu-tai-gia-lai-196250713124519597.htm
 Tiêu đề: Cán bộ công an tử vong vì tai nạn giao thông hy hữu tại Gia Lai
 Tóm tắt: (NLĐO) - Một vụ tai nạn giao thông hy hữu vừa xảy ra ở Gia Lai khiến một cán bộ công an tử vong tại chỗ khi đang đi bộ tập thể dục.

Toàn bộ dữ liệu: Cán bộ công an tử vong vì tai nạn giao thông hy hữu tại Gia Lai. (NLĐO) - Một vụ tai nạn giao thông hy hữu vừa xảy ra ở Gia Lai khiến một cán bộ công an tử vong tại chỗ khi đang đi bộ tập thể dục. Ngày 13-7, Phòng CSGT Công an tỉnh Gia Lai cho biết đang phối hợp điều tra nguyên nhân vụ tai nạn giao thông xảy ra vào khoảng 18 giờ 30 phút ngày 12-7, tại tuyến QL 19B thuộc phường An Nhơn Bắc. Hai bánh xe đầu kéo lăn ra ngoài trúng anh Phạm Hồng Vương Theo thông tin ban đầu, thời điểm trên, xe đầu kéo do ông Lê Đức Lợi (38 tuổi; ngụ phường Quy Nhơn Nam, tỉnh Gia Lai) điều khiển, lưu thông hướng từ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/Project_NLP/du_lieu_bao_nguoi_lao_dong/

In [ ]:
!cp /content/du_lieu_bao_nguoi_lao_dong/true.csv /content/drive/MyDrive/Project_NLP/du_lieu_bao_nguoi_lao_dong/